 # Sci/Tech News Q&A with Gemma (RAG)

**Track:** Build w Gemma — Gemma Challenge (Google Developer Group) · Hack Fiesta Miami, Aug 2026

This notebook implements a retrieval-augmented generation (RAG) pipeline:

1. Load the **AG News** dataset from Hugging Face and keep only the **Sci/Tech** label.
2. Chunk + embed the articles with a small sentence-embedding model.
3. Build a similarity index (FAISS) over the embeddings.
4. Retrieve the most relevant chunks for a user question.
5. Feed the retrieved context to **Gemma** (`google/gemma-2-2b-it`, via Hugging Face `transformers`) and generate a **cited** answer.

In [1]:
!pip install -q -U transformers accelerate datasets sentence-transformers faiss-cpu bitsandbytes "pyarrow>=19.0.0,<24"

## 1. Load AG News and keep only Sci/Tech

AG News labels: `0=World, 1=Sports, 2=Business, 3=Sci/Tech`.
We load it straight from the Hugging Face `datasets` hub (no manual download).


In [2]:
from datasets import load_dataset

N = 300
start = 79

dataset = load_dataset("fancyzhx/ag_news", split="train")
scitech_all = dataset.filter(lambda x: x["label"] == 3)

print(f"Start index: {start}  (rows {start} to {start + N - 1})")

scitech = scitech_all.select(range(start, start + N))

def split_title_body(example):
    parts = example["text"].split(" \\ ", 1)
    title = parts[0].strip()
    body = parts[1].strip() if len(parts) > 1 else example["text"]
    return {"title": title, "text": body}

scitech = scitech.map(split_title_body)

articles = [
    {"id": f"art-{i}", "title": r["title"], "text": r["text"]}
    for i, r in enumerate(scitech)
]

print(f"{len(articles)} articles loaded successfully starting from index {start}!")

Start index: 79  (rows 79 to 378)
300 articles loaded successfully starting from index 79!


In [3]:
def split_title_body(example):
    parts = example["text"].split(" \\ ", 1)
    title = parts[0].strip()
    body = parts[1].strip() if len(parts) > 1 else example["text"]
    return {"title": title, "body": body}

scitech = scitech.map(split_title_body)
articles = [
    {"id": f"art-{i}", "title": r["title"], "text": r["text"]}
    for i, r in enumerate(scitech)
]
articles[:3]

[{'id': 'art-0',
  'title': "Universities to Build Largest Telescope (AP) AP - The University of Texas and Texas A amp;M University are planning to construct what they call the world's largest telescope with the help of a Houston businessman's  #36;1.25 million gift.",
  'text': "Universities to Build Largest Telescope (AP) AP - The University of Texas and Texas A amp;M University are planning to construct what they call the world's largest telescope with the help of a Houston businessman's  #36;1.25 million gift."},
 {'id': 'art-1',
  'title': 'Mauritanian Capital Battles Locust Swarm (AP) AP - Residents burned tires and trash in the streets Thursday trying to drive off swarms of locusts descending on the region in what a U.N. agency called the worst sub-Saharan invasion in more than a decade.',
  'text': 'Mauritanian Capital Battles Locust Swarm (AP) AP - Residents burned tires and trash in the streets Thursday trying to drive off swarms of locusts descending on the region in what a 

## 2. Chunk + encoding the corpus

Articles here are already short, so "chunking" mostly means treating each article as one
chunk. We embed with `all-MiniLM-L6-v2` (fast, CPU-friendly) — **not** Gemma itself, since
Gemma is a text-generation model and doesn't expose an embeddings API. This mirrors the
production app's split between an embedding model and a generation model.

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

texts = [a["text"] for a in articles]
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.asarray(embeddings, dtype="float32")
print("Embedding matrix shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding matrix shape: (300, 384)


In [5]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dim}.")

FAISS index built with 300 vectors of dimension 384.


In [6]:
def retrieve(question: str, top_k: int = 5):
    q_emb = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        a = articles[idx]
        results.append({"title": a["title"], "text": a["text"], "score": float(score)})
    return results

for r in retrieve("AI chip funding announcement", top_k=3):
    print(f"{r['score']:.3f}  {r['title']}")

0.475  AMD starts shipping 90-nanometer chips The chipmaker appears to have met its most recent deadline for production of the smaller-featured chips.
0.389  iPod chipmaker plans stock offering PortalPlayer, whose chip powers Apple's MP3 player, files for IPO. Paperwork gives glimpse of potential evolution of iPod.
0.378  NASA Develops Robust Artificial Intelligence for Planetary Rovers NASA is planning to add a strong dose of artificial intelligence (AI) to planetary rovers to make them much more self-reliant, capable of making basic decisions during a mission. Scientists are developing very complex AI software that enables a higher level of robotic intelligence.


## 3. Configuring Gemma

`google/gemma-2-2b-it` is small enough to run on a single Kaggle T4 GPU in 4-bit
(via `bitsandbytes`). Swap in a larger variant (e.g. `google/gemma-2-9b-it`) if your
runtime has more memory.

In [7]:
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = userdata.get('gemma-rag-token')
login(token=hf_token)
print("Successfully logged in to Hugging Face Hub!")

GEMMA_MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print(f"Loaded {GEMMA_MODEL_ID} on {model.device}.")

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loaded google/gemma-2-2b-it on cuda:0.


In [8]:
def gemma_generate(system_prompt, user_prompt, max_new_tokens=256):

    chat = [
        {"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}
    ]

    formatted_prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = output_ids[0][input_len:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer

### 4. The RAG: retrieve → augment → generate (Gemma)

The system prompt forces Gemma to answer **only** from the numbered context and to
say so explicitly when the answer isn't in the retrieved articles — this is the
citation-or-refusal pattern used to keep answers grounded and reduce hallucination.

In [9]:
SYSTEM_PROMPT = (
    "You are a fact-checking assistant for Sci/Tech news. "
    "Answer ONLY using the numbered context snippets provided. "
    "Cite snippet numbers like [1], [2] that support each claim. "
    "If the context does not contain the answer, say so plainly instead of guessing."
)

def rag_answer(question: str, top_k: int = 5) -> dict:
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(f"[{i+1}] ({h['title']}) {h['text']}" for i, h in enumerate(hits))
    user_prompt = f"Context snippets from AG News (Sci/Tech):\n\n{context}\n\nQuestion: {question}\n\nAnswer:"
    answer = gemma_generate(SYSTEM_PROMPT, user_prompt)
    return {"question": question, "answer": answer, "sources": hits}

In [10]:
result = rag_answer("What recent Sci/Tech stories mention AI chips or semiconductors?")

print("Q:", result["question"])
print("\nGemma's answer:\n", result["answer"])
print("\nSources considered:")
for i, s in enumerate(result["sources"], 1):
    print(f"  [{i}] ({s['score']:.3f}) {s['title']}")

Q: What recent Sci/Tech stories mention AI chips or semiconductors?

Gemma's answer:
 Here are the Sci/Tech stories that mention AI chips or semiconductors:

1. **[3] NASA Develops Robust Artificial Intelligence for Planetary Rovers** - This snippet mentions "very complex AI software that enables a higher level of robotic intelligence" and "AI" in relation to planetary rovers.
2. **[4] Intel Shows Wireless Transceiver Prototype 90-nanometer radio chip promises more power than current CMOS technology.** - This snippet mentions a 90-nanometer radio chip and "Intel" and "wireless transceiver" which is a type of chip. 
3. **[1] (AMD starts shipping 90-nanometer chips** - This snippet mentions "90-nanometer chips" which are a type of semiconductor. 
 
Let me know if you'd like me to elaborate on any of these! 


Sources considered:
  [1] (0.390) AMD starts shipping 90-nanometer chips The chipmaker appears to have met its most recent deadline for production of the smaller-featured chips.
  [

In [11]:
for q in [
    "What are the trade-offs between two-phase commit protocols and event-driven eventual consistency in microservices databases?",
    "What are the best practices for implementing distributed tracing and correlation IDs across high-throughput microservices?",
    "How are distributed transactions and data consistency managed across independent microservices using the Saga pattern?",
    "How do circuit breaker patterns and rate limiters prevent cascading failures in a microservices architecture?",
    ]:
    r = rag_answer(q, top_k=4)
    print("="*80)
    print("Q:", r["question"])
    print(r["answer"])

Q: What are the trade-offs between two-phase commit protocols and event-driven eventual consistency in microservices databases?
The provided text snippets do not contain the answer to your question. 

Q: What are the best practices for implementing distributed tracing and correlation IDs across high-throughput microservices?
The provided context does not contain the answer to your question. 

Q: How are distributed transactions and data consistency managed across independent microservices using the Saga pattern?
The provided context does not contain the answer to your question. 

Q: How do circuit breaker patterns and rate limiters prevent cascading failures in a microservices architecture?
The provided context does not contain the answer to your question. 



## 5. Export the curated corpus for the companion Node.js app

The Node/Vercel app in this submission normally pulls AG News live from the Hugging Face
`datasets-server` REST API at request time. Exporting this notebook's already-filtered,
already-cleaned `articles` list lets the app **ingest this exact curated corpus** instead —
useful for reproducibility (the app serves precisely what was validated here) and for
offline/judging scenarios with unreliable Wi-Fi.


In [12]:
from datasets import load_dataset

print("Loading AG News dataset...")
dataset = load_dataset("sh0416/ag_news", split="train")

scitech_subset = dataset.filter(lambda x: x["label"] == 3).select(range(1000))

articles = []
for item in scitech_subset:
    # Find the first value in the item that is a string and contains text
    text_content = next((val for val in item.values() if isinstance(val, str) and len(val) > 20), "")
    if not text_content:
        continue

    articles.append({
        "title": text_content.split(".")[0],
        "text": text_content
    })

print(f"Articles collection with {len(articles)} items has been created well")

Loading AG News dataset...
Articles collection with 1000 items has been created well


In [13]:
import os

file_path = 'scitech_articles_export.json'
if os.path.exists(file_path):
    size = os.path.getsize(file_path)
    print(f"File size: {size} bytes")
else:
    print("File does not exist!")

File does not exist!


In [14]:
import json

scitech_data = [
    {"title": "Example Article", "content": "Sample content..."}
]


with open('scitech_articles_export.json', 'w', encoding='utf-8') as f:
    json.dump(scitech_data, f, ensure_ascii=False, indent=4)

print('Wrote data to scitech_articles_export.json')

Wrote data to scitech_articles_export.json


In [15]:
import json
from datasets import load_dataset
from IPython.display import FileLink

print("Loading AG News dataset from public mirror...")

dataset = load_dataset("nininiu/ag_news", split="train")

print("Filtering for Sci/Tech articles (label == 3)...")

scitech_subset = dataset.filter(lambda x: x["label"] == 3).select(range(1000))

articles = []
for item in scitech_subset:
    text_content = item.get("text") or item.get("description") or list(item.values())[0]
    if not isinstance(text_content, str) or len(text_content) < 10:
        continue

    articles.append({
        "title": text_content.split(".")[0],
        "text": text_content
    })

print(f"Loaded {len(articles)} genuine Sci/Tech articles successfully.")

export_filename = "scitech_articles_export.json"
with open(export_filename, "w", encoding="utf-8") as f:
    json.dump(articles, f, indent=2, ensure_ascii=False)

print(f"Successfully exported to {export_filename}")
display(FileLink(export_filename))

Loading AG News dataset from public mirror...
Filtering for Sci/Tech articles and excluding forbidden terms...


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]

Loaded 1000 clean Sci/Tech articles successfully.
Successfully exported to scitech_articles_export.json


/content/scitech_articles_export.json